In [57]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
import pandas as pd
import numpy as np

In [59]:
df = pd.read_csv("/content/drive/MyDrive/dirty_cafe_sales.csv")

In [60]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [61]:
df.shape

(10000, 8)

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [63]:
df.isnull().sum()

,0
Transaction ID,0
Item,333
Quantity,138
Price Per Unit,179
Total Spent,173
Payment Method,2579
Location,3265
Transaction Date,159


In [64]:
df.duplicated().sum()

np.int64(0)

In [65]:
(df == "ERROR").sum()

,0
Transaction ID,0
Item,292
Quantity,170
Price Per Unit,190
Total Spent,164
Payment Method,306
Location,358
Transaction Date,142


In [66]:
(df == "UNKNOWN").sum()

,0
Transaction ID,0
Item,344
Quantity,171
Price Per Unit,164
Total Spent,165
Payment Method,293
Location,338
Transaction Date,159


In [67]:
df_raw = df.copy()

In [68]:
df = df.replace(["ERROR","UNKNOWN"], np.nan)

In [69]:
df.isnull().sum()

,0
Transaction ID,0
Item,969
Quantity,479
Price Per Unit,533
Total Spent,502
Payment Method,3178
Location,3961
Transaction Date,460


In [70]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")

In [71]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              9031 non-null   object        
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    6822 non-null   object        
 6   Location          6039 non-null   object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 625.1+ KB


In [73]:
df.isnull().sum()

,0
Transaction ID,0
Item,969
Quantity,479
Price Per Unit,533
Total Spent,502
Payment Method,3178
Location,3961
Transaction Date,460


Total Spent = Quantity × Price Per Unit, so blanks in these columns were recovered by calculation rather than guessed.

In [74]:
df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])

In [75]:
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Total Spent"] / df["Quantity"])

In [76]:
df["Quantity"] = df["Quantity"].fillna(df["Total Spent"] / df["Price Per Unit"])

In [77]:
df[["Quantity", "Price Per Unit", "Total Spent"]].isnull().sum()

,0
Quantity,38
Price Per Unit,38
Total Spent,40


In [78]:
df.groupby("Item")["Price Per Unit"].unique()

,Price Per Unit
Item,
Cake,"[3.0, nan]"
Coffee,"[2.0, nan]"
Cookie,"[1.0, nan]"
Juice,"[3.0, nan]"
Salad,"[5.0, nan]"
Sandwich,"[4.0, nan]"
Smoothie,"[4.0, nan]"
Tea,"[1.5, nan]"


Each item has a single fixed price, so missing prices were filled using the item’s price from the lookup. Missing totals and quantities were then recalculated.

In [79]:
price_map = df.groupby("Item")["Price Per Unit"].first()

In [80]:
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Item"].map(price_map))

In [81]:
df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])
df["Quantity"] = df["Quantity"].fillna(df["Total Spent"] / df["Price Per Unit"])

In [82]:
df[["Quantity", "Price Per Unit", "Total Spent"]].isnull().sum()

,0
Quantity,23
Price Per Unit,6
Total Spent,23


In [83]:
price_to_item = {1.0: "Cookie", 1.5: "Tea", 2.0: "Coffee", 5.0: "Salad"}
df["Item"] = df["Item"].fillna(df["Price Per Unit"].map(price_to_item))

In [84]:
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Item"].map(price_map))
df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])
df["Quantity"] = df["Quantity"].fillna(df["Total Spent"] / df["Price Per Unit"])

In [85]:
df[["Item", "Quantity", "Price Per Unit", "Total Spent"]].isnull().sum()

,0
Item,480
Quantity,23
Price Per Unit,6
Total Spent,23


## Removing Unrecoverable Rows

26 rows (0.26%) where Quantity, Price Per Unit or Total Spent could not be recovered were removed. Dataset size: 10,000 → 9,974 rows.

In [86]:
df[["Quantity", "Price Per Unit", "Total Spent"]].isnull().any(axis=1).sum()

np.int64(26)

In [87]:
df = df.dropna(subset=["Quantity", "Price Per Unit", "Total Spent"])

In [88]:
df = df.reset_index(drop=True)

## Handling Missing Text Values
Payment Method (31.8% blank), Location (39.6%) and Item (4.8%) could not be recovered. Deleting these rows would have discarded valid sales, and imputing the most common value would have invented data. Blanks were labelled ‘Not Recorded’ to preserve all sales and make the data gap transparent. The remaining blank Items all had prices of 3.0 or 4.0, each shared by two products, so they were not guessed.

In [89]:
df.shape
df[["Quantity", "Price Per Unit", "Total Spent"]].isnull().sum()

,0
Quantity,0
Price Per Unit,0
Total Spent,0


In [90]:
df.shape

(9974, 8)

In [91]:
df["Payment Method"].value_counts(dropna=False)

,count
Payment Method,
NaN,3168
Digital Wallet,2284
Credit Card,2268
Cash,2254


In [92]:
df["Location"].value_counts(dropna=False)

,count
Location,
NaN,3952
Takeaway,3016
In-store,3006


In [93]:
df["Item"].value_counts(dropna=False)

,count
Item,
Coffee,1286
Salad,1270
Cookie,1211
Tea,1206
Juice,1168
Cake,1138
Sandwich,1127
Smoothie,1094
NaN,474


In [94]:
df[df["Item"].isnull()]["Price Per Unit"].value_counts()

,count
Price Per Unit,
3.0,247
4.0,227


In [95]:
df["Payment Method"] = df["Payment Method"].fillna("Not Recorded")
df["Location"] = df["Location"].fillna("Not Recorded")
df["Item"] = df["Item"].fillna("Not Recorded")

In [96]:
df.isnull().sum()

,0
Transaction ID,0
Item,0
Quantity,0
Price Per Unit,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,460


## Handling Missing Dates

460 rows (4.6%) have no Transaction Date. Transaction IDs are not in date order, so the date cannot be inferred from neighbouring rows, and imputing a single date would distort monthly and quarterly trends. These rows were kept because their Quantity, Price and Total Spent values are valid, and they are still needed for revenue totals. The dates were left blank (NaT) and will be excluded only from time-based analysis.

In [97]:
df["Transaction Date"].min()

Timestamp('2023-01-01 00:00:00')

In [98]:
df["Transaction Date"].max()

Timestamp('2023-12-31 00:00:00')

In [99]:
df["Transaction Date"].dt.year.value_counts()

,count
Transaction Date,
2023.0,9514


In [100]:
df["Transaction Date"].min()
df["Transaction Date"].max()
df["Transaction Date"].dt.year.value_counts()

,count
Transaction Date,
2023.0,9514


## Value Range Check

Summary statistics were reviewed for impossible values (negative or zero quantities or prices).

In [101]:
df.describe()

,Quantity,Price Per Unit,Total Spent,Transaction Date
count,9974.000000,9974.000000,9974.000000,9514
mean,3.024865,2.946962,8.927411,2023-07-01 23:14:35.593861888
min,1.000000,1.000000,1.000000,2023-01-01 00:00:00
25%,2.000000,2.000000,4.000000,2023-04-01 00:00:00
50%,3.000000,3.000000,8.000000,2023-07-02 00:00:00
75%,4.000000,4.000000,12.000000,2023-10-02 00:00:00
max,5.000000,5.000000,25.000000,2023-12-31 00:00:00
std,1.420504,1.280144,6.002644,NaN


In [102]:
df["Transaction ID"] = df["Transaction ID"].astype("string")

In [103]:
df["Transaction ID"].duplicated().sum()

np.int64(0)

In [104]:
(df["Total Spent"] != df["Quantity"] * df["Price Per Unit"]).sum()

np.int64(0)

## Duplicate Check

No duplicate rows were found (0 of 10,000), and all Transaction IDs are unique, so no rows were removed for duplication.

## Value Range Check

Summary statistics showed no impossible values: Quantity and Price Per Unit range from 1 to 5, Total Spent from 1 to 25, and all dates fall within 2023.

## Consistency Check

All Transaction IDs are unique, and Total Spent equals Quantity × Price Per Unit in every row.

In [105]:
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(col, "| lower:", lower, "| upper:", upper, "| outliers:", len(outliers))

Quantity | lower: -1.0 | upper: 7.0 | outliers: 0
Price Per Unit | lower: -1.0 | upper: 7.0 | outliers: 0
Total Spent | lower: -8.0 | upper: 24.0 | outliers: 268


In [106]:
df[df["Total Spent"] > 24]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
10,TXN_2548360,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-11-07
51,TXN_6342161,Salad,5.0,5.0,25.0,Not Recorded,Takeaway,2023-01-08
52,TXN_8914892,Salad,5.0,5.0,25.0,Digital Wallet,Not Recorded,2023-03-15
96,TXN_5220895,Salad,5.0,5.0,25.0,Cash,In-store,2023-06-10
100,TXN_9517146,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-10-30
...,...,...,...,...,...,...,...,...
9767,TXN_1232346,Salad,5.0,5.0,25.0,Digital Wallet,In-store,2023-01-04
9781,TXN_9506076,Salad,5.0,5.0,25.0,Not Recorded,Not Recorded,2023-01-17
9853,TXN_6393305,Salad,5.0,5.0,25.0,Digital Wallet,In-store,2023-10-19
9882,TXN_8922585,Salad,5.0,5.0,25.0,Credit Card,Not Recorded,2023-06-22


In [107]:
df.shape

(9974, 8)

In [108]:
df["Transaction Date"].isnull().sum()

np.int64(460)

## Before vs. After Cleaning Summary

The table below compares the raw dataset with the cleaned dataset.

In [109]:
placeholders = ["ERROR", "UNKNOWN"]

summary = pd.DataFrame({
    "Metric": ["Rows", "Blank (NaN) cells", "ERROR/UNKNOWN cells", "'Not Recorded' labels", "Duplicate rows", "Columns with correct data type"],
    "Before": [len(df_raw), df_raw.isnull().sum().sum(), df_raw.isin(placeholders).sum().sum(), 0, df_raw.duplicated().sum(), "4 of 8"],
    "After": [len(df), df.isnull().sum().sum(), df.isin(placeholders).sum().sum(), df.isin(["Not Recorded"]).sum().sum(), df.duplicated().sum(), "8 of 8"]
})
summary

,Metric,Before,After
0,Rows,10000,9974
1,Blank (NaN) cells,6826,460
2,ERROR/UNKNOWN cells,3256,0
3,'Not Recorded' labels,0,7594
4,Duplicate rows,0,0
5,Columns with correct data type,4 of 8,8 of 8


In [110]:
df.to_csv("/content/drive/MyDrive/cleaned_cafe_sales.csv", index=False)

## Standardisation Check

Value counts on Item, Payment Method and Location showed consistent spelling and capitalisation, with no stray spaces, so no text standardisation was needed. Fake missing values (ERROR, UNKNOWN) were standardised to true blanks, and dates were converted to datetime.

In [111]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9974 entries, 0 to 9973
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    9974 non-null   string        
 1   Item              9974 non-null   object        
 2   Quantity          9974 non-null   float64       
 3   Price Per Unit    9974 non-null   float64       
 4   Total Spent       9974 non-null   float64       
 5   Payment Method    9974 non-null   object        
 6   Location          9974 non-null   object        
 7   Transaction Date  9514 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(3), string(1)
memory usage: 623.5+ KB


## Conclusion

The raw dataset (10,000 rows) contained blanks, hidden placeholder values (ERROR and UNKNOWN) and every column stored as text. Placeholders were converted to true blanks, data types were corrected, and missing numeric values were recovered using Total Spent = Quantity × Price Per Unit and fixed per-item prices. 26 unrecoverable rows were removed. Missing categorical values were labelled "Not Recorded" and 460 missing dates were left blank to avoid distorting trends. No duplicates were found, and 268 high-value outliers were retained as valid orders. The cleaned dataset has 9,974 rows and is saved as cleaned_cafe_sales.csv.